# Retrieval analysis

I use this notebook to look inside the pipeline: what the parser makes of a messy PDF, what the chunks look like, how the four retrieval set-ups compare and where the system still fails. Everything runs offline on CPU in under a minute.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]
import pandas as pd
pd.set_option("display.max_colwidth", 110)
from copilot.parsing import parse_pdf
from copilot.chunking import chunk_document

## 1. Parsing one controlled document
The PDF has a running header, a footer with *Page x of y*, a title block table and a torque table with a wrapped cell. The parser recovers the metadata from the title block and turns the table into one line per row.

In [2]:
doc = parse_pdf(ROOT / "data/corpus/WI-ASM-014_RevC.pdf")
print(json.dumps(doc.meta, indent=1))
for s in doc.sections[:5]:
    print("\n##", s.heading, "(page", s.page, ")")
    for b in s.blocks:
        print("  ", b.kind, "|", b.text[:160].replace("\n", " / "))

{
 "doc_no": "WI-ASM-014",
 "revision": "C",
 "doc_type": "work instruction",
 "status": "current",
 "effective": "2026-06-01",
 "owner": "Production Engineering",
 "classification": "internal",
 "title": "Gearbox Housing Assembly - Cover and Flange Bolting"
}

## 1 Purpose (page 1 )
   text | This work instruction describes how the gearbox housing cover is sealed and bolted to the housing on assembly line L2. It applies to all e-drive variants ED-150

## 2 Safety (page 1 )
   text | Wear cut-resistant gloves when handling the housing cover. The flange edges are sharp. Wear nitrile gloves when working with the sealant (see HSE-POL-003). The 

## 3 Tools and materials (page 1 )
   table | Item: EC screwdriver; Specification: Station S4, program P-14 / Item: Sealant; Specification: Anaerobic flange sealant, type S-2 / Item: Dispensing unit; Specificat

## 4 Tightening torques (page 1 )
   text | All torques are final torques. Tolerance is the value given in the table. Bolts with pre-appl

## 2. Chunks

In [3]:
chunks = chunk_document(doc)
pd.DataFrame([{"chunk_id": c.chunk_id, "section": c.section, "page": c.page, "chars": len(c.text)} for c in chunks])

,chunk_id,section,page,chars
0,WI-ASM-014:C:000,1 Purpose,1,172
1,WI-ASM-014:C:001,2 Safety,1,221
2,WI-ASM-014:C:002,3 Tools and materials,1,266
3,WI-ASM-014:C:003,4 Tightening torques,1,535
4,WI-ASM-014:C:004,5 Procedure,1,997
5,WI-ASM-014:C:005,6 Deviations,2,224
6,WI-ASM-014:C:006,7 Records,2,120
7,WI-ASM-014:C:007,8 Revision history,2,260


In [4]:
print(chunks[3].index_text)

WI-ASM-014 Rev C - Gearbox Housing Assembly - Cover and Flange Bolting > 4 Tightening torques
All torques are final torques. Tolerance is the value given in the table. Bolts with pre-applied threadlocker must not be reused (see ES-TOR-001).
Joint: Housing cover bolts; Thread: M8; Torque: 25 Nm; Tolerance: +/- 2.5 Nm
Joint: Flange bolts to motor housing; Thread: M10; Torque: 45 Nm; Tolerance: +/- 4.5 Nm
Joint: Speed sensor bracket; Thread: M6; Torque: 9 Nm; Tolerance: +/- 0.9 Nm
Joint: Oil drain plug; Thread: M14x1.5; Torque: 30 Nm; Tolerance: +/- 3 Nm
The M10 flange torque was raised from 40 Nm to 45 Nm with ECN-2026-044.


## 3. Building the index
Same code path as the API: Qdrant (in memory here), WordLlama dense vectors and BM25 sparse vectors.

In [5]:
import time
from evaluate import build, eval_retrieval, roles_for
from copilot.evaluation import load_questions, is_relevant
t = time.time()
cp = build()
print(f"{cp.store.count()} chunks indexed in {time.time() - t:.1f} s")
corpus = pd.DataFrame(cp.store.scroll_all())
corpus.groupby(["doc_type"]).agg(documents=("doc_no", "nunique"), chunks=("chunk_id", "count"))

INFO:copilot.ingest:ingested ECN-2026-044 rev A: 6 chunks


INFO:copilot.ingest:ingested ES-TOR-001 rev D: 6 chunks


INFO:copilot.ingest:ingested HSE-POL-003 rev B: 7 chunks


INFO:copilot.ingest:ingested HSE-POL-007 rev A: 6 chunks


INFO:copilot.ingest:ingested MGT-REV-2026-Q3 rev A: 5 chunks


...

INFO:copilot.ingest:ingested WI-LOG-009 rev A: 6 chunks


INFO:copilot.ingest:ingested WI-MCH-022 rev A: 7 chunks


INFO:copilot.ingest:ingested WI-PNT-005 rev D: 7 chunks


138 chunks indexed in 2.9 s


,documents,chunks
doc_type,,
8d report,2,14
audit report,1,4
engineering change,1,6
engineering standard,1,6
hse policy,2,13
maintenance manual,2,12
maintenance procedure,1,6
meeting minutes,1,5
quality procedure,2,12


## 4. Retrieval comparison on the test questions

In [6]:
qs = load_questions(ROOT / "data/eval/test.jsonl")
summary, rows, leaks = eval_retrieval(cp, qs)
print("chunks returned that the user may not see:", leaks)
pd.DataFrame({k: v for k, v in summary.items() if not k.startswith("_")}).T

chunks returned that the user may not see: 0


,hit@1,hit@3,hit@5,hit@10,mrr,ndcg@10,latency_ms_p50
dense,0.6471,0.8151,0.8824,0.9832,0.7504,0.8046,5.2
bm25,0.8655,0.9412,0.9748,0.9832,0.9124,0.9284,5.8
hybrid (RRF),0.7059,0.9664,0.9832,1.0000,0.8305,0.8745,10.6
hybrid + rerank,0.9244,0.9916,1.0000,1.0000,0.9573,0.9671,21.0


In [7]:
pd.DataFrame(summary["_by_type_hybrid_rerank"]).T

,n,hit@1,hit@5
cross_doc,4.0,0.750,1.0
factual,54.0,0.963,1.0
numeric,12.0,1.000,1.0
procedural,12.0,0.917,1.0
table,33.0,0.879,1.0
version,4.0,0.750,1.0


BM25 alone is surprisingly strong on this corpus, because questions on the shop floor are full of exact tokens (M10, E-205, WI-PNT-005). The plain RRF fusion is worse than BM25 at rank 1, since the dense model pulls in chunks that are on-topic but not the answer. The reranker fixes most of that and pushes Hit@5 to 1.0.

## 5. What changes at rank 1 after reranking

In [8]:
df = pd.DataFrame(rows)
p = df.pivot_table(index="id", columns="config", values="hit@1")
changed = p[p["hybrid (RRF)"] != p["hybrid + rerank"]]
print("rerank fixed:", int((changed["hybrid + rerank"] > changed["hybrid (RRF)"]).sum()),
      "| rerank broke:", int((changed["hybrid + rerank"] < changed["hybrid (RRF)"]).sum()))
qmap = {q.id: q.question for q in qs}
changed.assign(question=[qmap[i] for i in changed.index])[["hybrid (RRF)", "hybrid + rerank", "question"]].head(15)

rerank fixed: 27 | rerank broke: 1


config,hybrid (RRF),hybrid + rerank,question
id,,,
t005,0.0,1.0,How wide must the sealant bead on the gearbox flange be?
t007,0.0,1.0,How long does the sealant need to cure before the leak test?
t018,0.0,1.0,What is the tolerance of the bearing seat diameter on the rotor shaft RS-40?
t020,0.0,1.0,What is the maximum runout of the rotor shaft?
t022,0.0,1.0,What happens if an in-process measurement on the rotor shaft is out of specification?
t025,0.0,1.0,What is the reject limit of the gearbox leak test?
t026,0.0,1.0,How long is the settle time in the leak test?
t028,0.0,1.0,What do I do when a housing fails the leak test twice?
t039,0.0,1.0,What are the steps of lockout tagout?


## 6. Remaining retrieval misses at rank 1

In [9]:
def top_hits(q, k=3):
    r = cp.retriever.retrieve(q.question, roles_for(cp, q.role), top_k=k)
    return [(c.payload["chunk_id"], c.payload["section"], round(c.score, 2), is_relevant(c.payload, q.gold)) for c in r.chunks]
miss = [q for q in qs if q.answerable and not is_relevant(cp.retriever.retrieve(q.question, roles_for(cp, q.role), top_k=1).chunks[0].payload, q.gold)]
for q in miss:
    print(q.id, q.question, "| gold:", q.gold)
    for h in top_hits(q): print("     ", h)

t024 What is the test pressure on the leak test bench? | gold: ['WI-PNT-005:4']
      ('WI-PNT-005:D:000', '1 Purpose', 1.38, False)
      ('WI-PNT-005:D:005', '6 Records', 1.26, False)
      ('WI-PNT-005:D:003', '4 Test parameters', 1.13, True)
t047 What do I do if a large amount of oil gets into a drain? | gold: ['HSE-POL-003:5']
      ('WI-ASM-014:C:004', '5 Procedure', 0.84, False)
      ('WI-ASM-014:C:003', '4 Tightening torques', 0.65, False)
      ('MP-LOTO-002:C:002', '3 The six steps', 0.6, False)
t078 What hipot voltage is used at the end-of-line test? | gold: ['TS-EOL-012:2']
      ('TS-EOL-012:B:005', '6 Revision history', 1.31, False)
      ('TS-EOL-012:B:001', '2 Electrical safety tests', 1.16, True)
      ('TS-EOL-012:B:000', '1 Scope', 1.03, False)
t084 Which tightening method is required for class A joints? | gold: ['ES-TOR-001:2']
      ('ES-TOR-001:D:002', '3 Tolerances', 1.2, False)
      ('ES-TOR-001:D:001', '2 Joint classes', 1.16, True)
      ('ES-TOR-001:D:005',

## 7. Answers and abstention

In [10]:
ans = pd.read_csv(ROOT / "reports/answers_test.csv")
neg = ans[~ans.answerable]
print("restricted/unanswerable questions answered by mistake:", int((~neg.abstained).sum()))
pos = ans[ans.answerable]
print("answerable but abstained:", int(pos.abstained.sum()), "of", len(pos))
pos[pos.abstained][["id", "type", "top_score", "question"]]

restricted/unanswerable questions answered by mistake:

 0
answerable but abstained: 14 of 119


,id,type,top_score,question
6,t007,numeric,1.0732,How long does the sealant need to cure before the leak test?
12,t013,factual,1.0160,What colour paint pen is used to mark the bolts after tightening?
29,t030,numeric,1.0265,What resistance range does the wrist strap tester accept?
31,t032,factual,0.9733,An inverter board fell on the floor but looks fine. Can I still use it?
33,t034,table,0.8218,How many KLT boxes can be stacked on top of each other?
39,t040,factual,0.8950,How often is the LOTO refresher training required?
46,t047,procedural,0.8371,What do I do if a large amount of oil gets into a drain?
48,t049,factual,0.9101,How much hazardous substance may be kept at the workplace?
65,t066,table,0.9865,How often is the coolant replaced completely on the machining centres?
79,t080,factual,0.9827,How long are EOL test results stored?


In [11]:
wrong = pos[(~pos.abstained) & (pos.facts_ok != True)]
wrong[["id", "type", "question", "answer"]]

,id,type,question,answer
21,t022,procedural,What happens if an in-process measurement on the rotor shaft is out of specification?,- Characteristic: Bearing seat diameter; Specification: 40 k6 (+0.018 / +0.002 mm); Frequency: every 25th ...
85,t086,factual,Why was the gearbox flange torque changed in ECN-2026-044?,"- Rev: C; Date: 2026-06-01; Change: M10 flange torque 40 -> 45 Nm, sealant bead 2.5 -> 3.0 mm, open time 1..."
89,t090,factual,What was the root cause of the oil leakage complaint on the gearbox flange?,"- Repeated E-205 alarms had been acknowledged without venting the pump. [1] - In addition, the M10 flange ..."
94,t095,factual,Why were the rotor shaft bearing seats on MC-504 out of tolerance?,"- During the final inspection on 2026-09-03, 38 rotor shafts RS-40 from machine MC-504 had a bearing seat ..."


The wrong extractive answers are mostly *why* questions (the right section is found, but the chosen sentence is the symptom and not the cause) and questions where the key sentence sits next to a better-matching one. This is exactly what the LLM mode is for; the Colab notebook `02_llm_evaluation_colab.ipynb` measures it.